# Data Mining Assignment: Apache Spark Data Lake Preprocessing
**Name:** Maneesh Kumar  
**Roll No:** LIT2023051

In [1]:
!pip install -q pyspark

In [2]:
import os
import glob
import shutil
import subprocess

os.makedirs("data_lake/bronze", exist_ok=True)
os.makedirs("data_lake/silver/preprocessed_sales", exist_ok=True)

repo_dir = "data-mining-data-lake-assignment"
if not os.path.exists(repo_dir):
    subprocess.run(
        ["git", "clone", "-q", "https://github.com/necromancersanju-netizen/data-mining-data-lake-assignment.git"],
        check=True
    )

matches = glob.glob(f"{repo_dir}/**/sales.csv", recursive=True)
if not matches:
    raise FileNotFoundError("sales.csv not found in cloned repository. Upload sales.csv to data_lake/bronze/sales.csv.")

shutil.copy2(matches[0], "data_lake/bronze/sales.csv")

for root, dirs, files in os.walk("data_lake"):
    level = root.replace("data_lake", "").count(os.sep)
    indent = " " * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    for f in files:
        print(f"{indent}  {f}")

data_lake/
  silver/
    preprocessed_sales/
  bronze/
    sales.csv


In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType

spark = (
    SparkSession.builder
    .appName("DataLakePreprocessing_LIT2023051")
    .config("spark.sql.ansi.enabled", "false")
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY")
    .getOrCreate()
)

bronze_path = "data_lake/bronze/sales.csv"
silver_dir = "data_lake/silver/preprocessed_sales"
silver_single_csv = "data_lake/silver/sales_silver.csv"

df_bronze = spark.read.option("header", True).option("inferSchema", True).csv(bronze_path)
df_raw = spark.read.option("header", True).option("inferSchema", False).csv(bronze_path)

df_bronze.printSchema()
df_bronze.show(10, truncate=False)

bronze_count = df_bronze.count()
print(f"Total Bronze record count: {bronze_count}")

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- discount_percent: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = true)

+--------+-----------+---------------+---------------------+---------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name  |product              |category |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|
+--------+-----------+---------------+---------------------+---------+--------+----------+--------

In [4]:
cols = df_raw.columns
cols_lower = {c: c.strip().lower().replace(" ", "_") for c in cols}

def find_col(keywords):
    for orig, norm in cols_lower.items():
        if any(k in norm for k in keywords):
            return orig
    return None

qty_col = find_col(["quantity", "qty"])
price_col = find_col(["unit_price", "unitprice", "price"])
discount_col = find_col(["discount"])
date_cols = [orig for orig, norm in cols_lower.items() if "date" in norm]

numeric_special = {c for c in [qty_col, price_col, discount_col] if c is not None}
id_cols = {orig for orig, norm in cols_lower.items() if norm == "id" or norm.endswith("_id") or norm.endswith("id")}
text_cols = [c for c in cols if c not in numeric_special and c not in date_cols]

# 1. Exact duplicate rows
distinct_count = df_raw.distinct().count()
duplicate_count = bronze_count - distinct_count
print(f"Exact duplicate rows: {duplicate_count}")

# 2. Null or blank values per column
null_exprs = [
    F.sum(F.when(F.col(c).isNull() | (F.trim(F.col(c)) == ""), 1).otherwise(0)).alias(c)
    for c in cols
]
print("\nNull / blank counts per column:")
df_raw.select(null_exprs).show(truncate=False)

# 3. Text-format inconsistencies (extra whitespace or inconsistent casing)
print("Text-format inconsistencies (extra whitespace or mixed casing):")
for c in text_cols:
    if c in id_cols:
        ws_cnt = df_raw.filter(F.col(c).isNotNull() & (F.col(c) != F.trim(F.col(c)))).count()
        print(f"  {c}: leading/trailing whitespace={ws_cnt}")
    else:
        ws_cnt = df_raw.filter(
            F.col(c).isNotNull() & (F.col(c) != F.regexp_replace(F.trim(F.col(c)), r"\s+", " "))
        ).count()
        case_variants = (
            df_raw.filter(F.col(c).isNotNull() & (F.trim(F.col(c)) != ""))
            .groupBy(F.lower(F.trim(F.col(c))).alias("norm_val"))
            .agg(F.countDistinct(F.trim(F.col(c))).alias("variants"))
            .filter(F.col("variants") > 1)
            .count()
        )
        print(f"  {c}: extra_whitespace_rows={ws_cnt}, inconsistent_casing_groups={case_variants}")

# 4. Invalid numerical values (non-numeric, <= 0 for qty/price, < 0 or > 100 for discount)
print("\nInvalid numerical values:")
if qty_col:
    qty_num = F.col(qty_col).cast(DoubleType())
    invalid_qty = df_raw.filter(
        F.col(qty_col).isNotNull() & (F.trim(F.col(qty_col)) != "") & (qty_num.isNull() | (qty_num <= 0))
    ).count()
    print(f"  {qty_col} (non-numeric or <= 0): {invalid_qty}")

if price_col:
    price_clean = F.regexp_replace(F.trim(F.col(price_col)), r"[^0-9.\-]", "").cast(DoubleType())
    invalid_price = df_raw.filter(
        F.col(price_col).isNotNull() & (F.trim(F.col(price_col)) != "") & (price_clean.isNull() | (price_clean <= 0))
    ).count()
    print(f"  {price_col} (non-numeric or <= 0): {invalid_price}")

if discount_col:
    disc_clean = F.regexp_replace(F.trim(F.col(discount_col)), r"%", "").cast(DoubleType())
    invalid_disc = df_raw.filter(
        F.col(discount_col).isNotNull() & (F.trim(F.col(discount_col)) != "") & (disc_clean.isNull() | (disc_clean < 0) | (disc_clean > 100))
    ).count()
    print(f"  {discount_col} (non-numeric, < 0, or > 100): {invalid_disc}")

# 5. Inconsistent and invalid dates
date_formats = [
    "yyyy-MM-dd", "dd-MM-yyyy", "MM-dd-yyyy",
    "yyyy/MM/dd", "dd/MM/yyyy", "MM/dd/yyyy",
    "yyyy.MM.dd", "dd.MM.yyyy",
    "dd MMM yyyy", "MMM dd, yyyy", "MMMM dd, yyyy",
    "yyyy-MM-dd HH:mm:ss", "dd/MM/yyyy HH:mm:ss", "MM/dd/yyyy HH:mm:ss"
]

def parse_any_date(col_expr):
    trimmed = F.trim(col_expr)
    return F.coalesce(*[F.to_date(trimmed, fmt) for fmt in date_formats])

print("\nDate inconsistencies & invalid dates:")
for dc in date_cols:
    parsed = parse_any_date(F.col(dc))
    invalid_date_cnt = df_raw.filter(
        F.col(dc).isNotNull() & (F.trim(F.col(dc)) != "") & parsed.isNull()
    ).count()
    non_iso_cnt = df_raw.filter(
        F.col(dc).isNotNull() & parsed.isNotNull() & ~F.trim(F.col(dc)).rlike(r"^\d{4}-\d{2}-\d{2}$")
    ).count()
    print(f"  {dc}: non_standard_format_rows={non_iso_cnt}, invalid_date_rows={invalid_date_cnt}")

Exact duplicate rows: 10

Null / blank counts per column:
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|order_id|customer_id|customer_name|product|category|quantity|unit_price|discount_percent|payment_method|city|state|order_date|order_status|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|0       |0          |10           |0      |0       |0       |0         |0               |8             |8   |0    |0         |0           |
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+

Text-format inconsistencies (extra whitespace or mixed casing):
  order_id: leading/trailing whitespace=0
  customer_id: leading/trailing whitespace=0
  customer_name: extra_whitespace_rows=3, inconsistent_casing_groups=0
  product: ext

In [5]:
df_silver = df_raw

# 1. Remove exact duplicate rows
df_silver = df_silver.dropDuplicates()

# Convert blank strings to null across all columns
for c in cols:
    df_silver = df_silver.withColumn(
        c,
        F.when(F.trim(F.col(c)) == "", F.lit(None)).otherwise(F.col(c))
    )

# 2. Trim extra spaces and standardize capitalization in text fields
for c in text_cols:
    cleaned = F.regexp_replace(F.trim(F.col(c)), r"\s+", " ")
    if c in id_cols:
        df_silver = df_silver.withColumn(c, F.upper(cleaned))
    elif "email" in cols_lower[c]:
        df_silver = df_silver.withColumn(c, F.lower(cleaned))
    else:
        df_silver = df_silver.withColumn(c, F.initcap(cleaned))

# 3. Validate quantity, unit price, and discount percentage
if qty_col:
    qty_num = F.col(qty_col).cast(DoubleType())
    df_silver = df_silver.withColumn(
        qty_col,
        F.when(qty_num.isNotNull() & (qty_num > 0), qty_num.cast("int")).otherwise(F.lit(None))
    )

if price_col:
    price_num = F.regexp_replace(F.trim(F.col(price_col)), r"[^0-9.\-]", "").cast(DoubleType())
    df_silver = df_silver.withColumn(
        price_col,
        F.when(price_num.isNotNull() & (price_num > 0), F.round(price_num, 2)).otherwise(F.lit(None))
    )

if discount_col:
    disc_num = F.regexp_replace(F.trim(F.col(discount_col)), r"%", "").cast(DoubleType())
    df_silver = df_silver.withColumn(
        discount_col,
        F.when(disc_num.isNotNull() & (disc_num >= 0) & (disc_num <= 100), F.round(disc_num, 2)).otherwise(F.lit(0.0))
    )

# 4. Standardize valid dates to yyyy-MM-dd and set invalid dates to NULL
for dc in date_cols:
    df_silver = df_silver.withColumn(dc, F.date_format(parse_any_date(F.col(dc)), "yyyy-MM-dd"))

# 5. Handle missing values while preserving useful records
df_silver = df_silver.dropna(how="all")
critical_cols = [c for c in [qty_col, price_col] if c is not None]
if critical_cols:
    df_silver = df_silver.dropna(subset=critical_cols)

fill_text_cols = [c for c in text_cols if c not in id_cols]
if fill_text_cols:
    df_silver = df_silver.fillna({c: "Unknown" for c in fill_text_cols})

# Drop any newly formed exact duplicates after text/date standardization
df_silver = df_silver.dropDuplicates()

# Save to Silver folder in CSV format
df_silver.write.mode("overwrite").option("header", True).csv(silver_dir)

# Also save a single consolidated CSV for GitHub submission (silver/sales_silver.csv)
temp_single_dir = "data_lake/silver/_tmp_single"
df_silver.coalesce(1).write.mode("overwrite").option("header", True).csv(temp_single_dir)
part_files = glob.glob(os.path.join(temp_single_dir, "part-*.csv"))
shutil.copy2(part_files[0], silver_single_csv)
shutil.rmtree(temp_single_dir)

print(f"Saved Silver dataset to: {silver_dir}")
print(f"Saved single Silver CSV to: {silver_single_csv}")

Saved Silver dataset to: data_lake/silver/preprocessed_sales
Saved single Silver CSV to: data_lake/silver/sales_silver.csv


In [6]:
df_silver_verify = spark.read.option("header", True).option("inferSchema", True).csv(silver_dir)
silver_count = df_silver_verify.count()

print(f"Row count before preprocessing (Bronze): {bronze_count}")
print(f"Row count after preprocessing  (Silver): {silver_count}")
print(f"Rows removed: {bronze_count - silver_count}")

df_silver_verify.printSchema()
df_silver_verify.show(10, truncate=False)

print("Preprocessing operations performed:")
print("1. Removed exact duplicate rows.")
print("2. Converted blank strings to nulls, dropped rows with missing/invalid critical numeric values (quantity/unit price), and filled missing categorical text fields with 'Unknown'.")
print("3. Trimmed extra spaces and standardized text capitalization (initcap for text fields, uppercase for IDs).")
print("4. Validated quantity (> 0), unit price (> 0), and discount percentage (0 to 100; invalid/missing discounts set to 0.0).")
print("5. Standardized valid dates into yyyy-MM-dd format and set clearly invalid date strings to NULL.")
print("6. Preserved valid records and saved the Silver output as CSV without altering the Bronze layer.")

Row count before preprocessing (Bronze): 1000
Row count after preprocessing  (Silver): 970
Rows removed: 30
root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount_percent: double (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = true)

+--------+-----------+------------------+----------------------+---------------+--------+----------+----------------+----------------+-------+-------------+----------+------------+
|order_id|customer_id|customer_name     |product               |category       |quantity|unit_price|discount_percent|payment_method  |city   |state        |order